# Spark Bronze wikpedia page reads

In [1]:
exeuction_date = "2025-01-01"
full_refresh = True

In [2]:
import os
import requests
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [3]:

def get_config(config_name):

    config_server_url = os.environ.get("TFDS_CONFIG_URL")
    if config_server_url is None:
        config_server_url = "http://tfds-config:8005/api/configs"

    config_url = config_server_url + "/" + config_name

    print(f"retrieving {config_name} config from {config_url}")
    response = requests.get(config_url)
    response.raise_for_status()
    if response.json() is None:
        raise ValueError(f"Config '{config_name}' not found. config server response: {response.text}")
    cfg = response.json().get("config")
    if cfg is None:
        raise ValueError(f"Config '{config_name}' does not have a 'config' key. Config server response: {response.text}")

    if config_name=='s3' and "TFDS_S3_URL" in os.environ.keys():
        cfg["url"] = os.environ["TFDS_S3_URL"]
    if config_name=='spark' and "TFDS_SPARK_MASTER_URL" in os.environ.keys():
        cfg["master_url"] = os.environ["TFDS_SPARK_MASTER_URL"]
    return cfg


def get_spark_session():
    """Get spark client for s3."""
    s3_cfg = get_config("s3")
    spark_cfg = get_config("spark")

    print(f"using s3 endpoint: {s3_cfg['url']}")
    print(f"using spark master: {spark_cfg['master_url']}")

    spark_session = (  SparkSession
        .builder
        .master('spark://spark-master:7077')
        .appName("Wikipedia page reads - Bronze")
        .config("spark.hadoop.fs.s3a.access.key", s3_cfg["access_key"])
        .config("spark.hadoop.fs.s3a.secret.key", s3_cfg["secret_key"])
        .config("spark.hadoop.fs.s3a.endpoint", 'http://s3-ninja:9000/s3')
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config(
            "spark.jars",
            "../jars/hadoop-aws-3.3.4.jar,../jars/aws-java-sdk-bundle-1.12.262.jar")
        .getOrCreate()
    )

    return spark_session


In [ ]:
spark = get_spark_session()


s3_path = "s3a://data/wikipedia_pageviews/2025/2025-01/10/*.gz"
s3_path = "s3a://data/wikipedia_pageviews/2024/2024-12/31/pageviews-20241231-000000.gz"
s3_path = "s3a://data/test_small"

schema = StructType([
    StructField(name="domain_code", dataType=StringType(), nullable = False),
    StructField("page_title", StringType(), False),
    StructField("count_views", StringType(), False),
    StructField("total_response_size", StringType(), False),
])
# spark.sparkContext.setLogLevel("INFO")
print(f"Reading data from {s3_path}")

df = (
    spark.read.format("csv")
    .option("header", "false")
    .option("inferSchema", "false")
    .schema(schema)
    .load(s3_path)
)

# Show the first few rows of the DataFrame
print("Data from S3:")
try:
    df.show()
except Exception as e:
    print(f"Error displaying DataFrame: {e}")


spark.stop()

retrieving s3 config from http://tfds-config:8005/api/configs/s3
retrieving spark config from http://tfds-config:8005/api/configs/spark
using s3 endpoint: http://s3-ninja:9000/s3
using spark master: spark://spark-master:7077


25/04/07 16:05:36 WARN Utils: Your hostname, McJens.local resolves to a loopback address: 127.0.0.1; using 192.168.0.152 instead (on interface en0)
25/04/07 16:05:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
25/04/07 16:05:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Reading data from s3a://data/test_small


25/04/07 16:05:41 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Data from S3:


+-----------------------------+--------------------+-----------+-------------------+
|                  domain_code|          page_title|count_views|total_response_size|
+-----------------------------+--------------------+-----------+-------------------+
|         "" Category:Candi...|                NULL|       NULL|               NULL|
|         "" Category:FAQ/e...|                NULL|       NULL|               NULL|
|         "" Category:Hebre...|                NULL|       NULL|               NULL|
|         "" Category:Lists...|                NULL|       NULL|               NULL|
|         "" Category:Wikid...|                NULL|       NULL|               NULL|
|         "" "File:God_and_...|\"_(IA_cu31924029...|       NULL|               NULL|
|         "" File:M._Marion...|                NULL|       NULL|               NULL|
|         "" File:NLC403-31...|                NULL|       NULL|               NULL|
|"" File:中华人民共和国全国...|                NULL|       NULL|          

In [ ]:
# import os
# import os
# import requests
# import boto3
# from pyspark.sql import SparkSession
# from pyspark.sql.types import StructType, StructField, StringType, IntegerType
# _config_server_url = None
# os.environ["TFDS_CONFIG_URL"] = 'http://127.0.0.1:8005/api/configs'
# _spark_session = None
# os.environ["TFDS_SPARK_MASTER_URL"] = 'spark://127.0.0.1:7077'
# os.environ["TFDS_S3_URL"] = 'http://127.0.0.1:8004/s3'


# def get_config(config_name):
#     global _config_server_url
#     if _config_server_url is None:
#         _config_server_url = os.environ.get("TFDS_CONFIG_URL")
#         if _config_server_url is None:
#             _config_server_url = "http://tfds-config:8005/api/configs"
#             print(f"TFDS_CONFIG_URL not set, using default: {_config_server_url}")
#         else:
#             print(f"using TFDS_CONFIG_URL: {_config_server_url}")
#     config_url = _config_server_url + "/" + config_name

#     print(f"retrieving {config_name} config from {config_url}")
#     response = requests.get(config_url)
#     response.raise_for_status()
#     if response.json() is None:
#         raise ValueError(f"Config '{config_name}' not found. config server response: {response.text}")
#     cfg = response.json().get("config")
#     if cfg is None:
#         raise ValueError(f"Config '{config_name}' does not have a 'config' key. Config server response: {response.text}")

#     if config_name=='s3' and "TFDS_S3_URL" in os.environ.keys():
#         cfg["url"] = os.environ["TFDS_S3_URL"]
#     if config_name=='spark' and "TFDS_SPARK_MASTER_URL" in os.environ.keys():
#         cfg["master_url"] = os.environ["TFDS_SPARK_MASTER_URL"]
#     return cfg

# def test_spark():


#     schema = StructType([
#         StructField(name="domain_code", dataType=StringType(), nullable = False),
#         StructField("page_title", StringType(), False),
#         StructField("count_views", StringType(), False),
#         StructField("total_response_size", StringType(), False),
#     ])

#     s3_cfg = get_config("s3")

#     spark = (  SparkSession
#         .builder
#         .master('spark://spark-master:7077')
#         .appName("Wikipedia reads Bronze")
#         .config("spark.hadoop.fs.s3a.access.key", s3_cfg["access_key"])
#         .config("spark.hadoop.fs.s3a.secret.key", s3_cfg["secret_key"])
#         .config("spark.hadoop.fs.s3a.endpoint", 'http://s3-ninja:9000/s3')
#         .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
#         .config("spark.hadoop.fs.s3a.path.style.access", "true")
#         .config(
#             "spark.jars",
#             "../jars/hadoop-aws-3.3.4.jar,../jars/aws-java-sdk-bundle-1.12.262.jar")
#         .getOrCreate()
#     )

#     # spark = (
#     #     SparkSession.builder
#     #     .master('local[*]')
#     #     .appName("Wikipedia reads Bronze")
#     #     .getOrCreate()
#     # )
#     spark.sparkContext.setLogLevel("WARN")

#     s3_path = "s3a://data/test_small"
#     s3_path = "/tmp/data/test_small"

#     s3_path = "/tmp/data/s3/data/*.gz"
#     s3_path = "s3a://data/test_small"
#     s3_path = "s3a://data/wikipedia_pageviews/2024/2024-12/31/pageviews-20241231-000000.gz"

#     df = (
#         spark.read.format("csv")
#         .option("header", "false")
#         .option("inferSchema", "true")
#         .option("delimiter", " ")
#         .schema(schema)
#         .load(s3_path)
#     )

#     # Show the first few rows of the DataFrame
#     print("Data from S3:")
#     try:

#         df.show(5)
#         pass
#     except Exception as e:
#         print(f"Error displaying DataFrame: {e}")
#     spark.stop()
# test_spark()